# Ensemble Learning - Stacking Companion Notebook

This notebook is the practical code companion for the stacking part of [`lecture_notes/12_ensemble_learning.pdf`](../../lecture_notes/12_ensemble_learning.pdf). The lecture notes explain why out-of-fold predictions are essential; the cells below show how that principle appears in a concrete `scikit-learn` workflow.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 42


This setup cell imports the heterogeneous models used to form the stacking ensemble. Diversity matters here: the whole point of stacking is to combine base learners that capture different aspects of the data rather than multiple copies of essentially the same model.


## Define Diverse Base Learners and the Meta-Model

The base layer mixes tree-based and margin-based models, while the meta-model is kept simple.


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

base_learners = [
    ("rf", RandomForestClassifier(n_estimators=150, random_state=SEED)),
    ("gb", GradientBoostingClassifier(n_estimators=120, random_state=SEED)),
    ("svc", make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEED))),
]

meta_learner = LogisticRegression(max_iter=5000)

stacking = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,
    stack_method="predict_proba",
)


This cell fixes the train/test split and builds the stacking architecture. The important implementation detail is `cv=5`: this tells `StackingClassifier` to create the out-of-fold predictions needed to train the meta-model without leakage.


In [ ]:
comparison_models = {
    "Random Forest": base_learners[0][1],
    "Gradient Boosting": base_learners[1][1],
    "SVC": base_learners[2][1],
    "Stacking": stacking,
}

cv_rows = []
for name, model in comparison_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    cv_rows.append({
        "model": name,
        "cv_mean_accuracy": scores.mean(),
        "cv_std": scores.std(),
    })

pd.DataFrame(cv_rows).sort_values("cv_mean_accuracy", ascending=False).round(4)


This benchmark compares each base learner against the final stacking ensemble under the same cross-validation protocol. The question is not whether stacking must always win, but whether combining complementary learners improves over the best individual model on this task.


## Final Test Comparison

After the cross-validation comparison, we fit each model on the full training split and evaluate once on held-out test data.


In [ ]:
test_rows = []
for name, model in comparison_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    test_rows.append({
        "model": name,
        "test_accuracy": accuracy_score(y_test, y_pred),
    })

pd.DataFrame(test_rows).sort_values("test_accuracy", ascending=False).round(4)


This table is the clean held-out comparison. Read it together with the cross-validation table above: if stacking performs well in both places, that is stronger evidence than a single isolated test result.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.ravel()

for ax, (name, model) in zip(axes, comparison_models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax, colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.show()


The confusion matrices reveal whether the ensemble changes the type of errors being made, not just the headline accuracy. This matters because stacking is supposed to exploit complementary strengths; the confusion matrices make that complementarity easier to inspect.


## Inspect the Meta-Features Used by the Final Estimator

One way to make stacking less opaque is to look at the probabilities produced by the base layer.


In [ ]:
rf_fitted = comparison_models["Random Forest"]
gb_fitted = comparison_models["Gradient Boosting"]
svc_fitted = comparison_models["SVC"]

meta_features = pd.DataFrame({
    "rf_p(class=1)": rf_fitted.predict_proba(X_test[:10])[:, 1],
    "gb_p(class=1)": gb_fitted.predict_proba(X_test[:10])[:, 1],
    "svc_p(class=1)": svc_fitted.predict_proba(X_test[:10])[:, 1],
    "true_label": y_test[:10],
})
meta_features.round(4)


This table is a small window into what the meta-model sees. Each row corresponds to one example, and each probability column corresponds to one base learner’s opinion. If the base models disagree in informative ways, stacking has something useful to learn; if they always agree, the meta-model has much less room to add value.


In [ ]:
stacking_meta = comparison_models["Stacking"].final_estimator_
pd.Series(
    stacking_meta.coef_.ravel(),
    index=["rf_p(class=1)", "gb_p(class=1)", "svc_p(class=1)"],
    name="meta_model_weight",
).sort_values(ascending=False).round(4)


The fitted logistic meta-model assigns a weight to each base learner’s probability. This is not the whole story, but it is a useful rough summary of which base predictions are contributing more strongly to the final ensemble decision.


## Why Diversity Matters

A simple way to check whether stacking has useful diversity is to look at the correlation between the base learners’ predicted probabilities.


In [ ]:
base_proba_df = pd.DataFrame({
    "Random Forest": rf_fitted.predict_proba(X_test)[:, 1],
    "Gradient Boosting": gb_fitted.predict_proba(X_test)[:, 1],
    "SVC": svc_fitted.predict_proba(X_test)[:, 1],
})
base_proba_df.corr().round(3)


In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(base_proba_df.corr(), cmap="Blues", vmin=0, vmax=1)
plt.xticks(range(3), base_proba_df.columns, rotation=20)
plt.yticks(range(3), base_proba_df.columns)
plt.colorbar(label="correlation")
plt.title("Correlation between base-model probabilities")
plt.tight_layout()
plt.show()


This final diagnostic connects the implementation back to the theory. Stacking benefits most when the base learners are not redundant. Very high correlation means the models are mostly echoing each other; lower but still meaningful agreement means the meta-model has a chance to exploit complementary information.
